# 05 — Basic RAG for CA Use Cases

> **ICAN CA Training — Generative AI & RAG (6 hours).** This notebook is part of a 14-notebook curriculum. All data is synthetic. Confidential client data must not be used with public APIs without engagement-letter authority. AI output must always be verified by a qualified professional.


## Learning objectives
1. Understand the RAG pipeline: load → chunk → embed → retrieve → prompt → answer.
2. Build a basic RAG system over the synthetic CA documents.
3. Ask CA-specific questions and read grounded answers.


## The RAG idea in one paragraph

Pure LLMs answer from memory — sometimes brilliantly, sometimes by hallucinating. **RAG (Retrieval-Augmented Generation)** changes that: before answering, we *retrieve* the most relevant chunks from our own documents, paste them into the prompt, and ask the model to answer **only from those chunks**. Result: grounded answers with sources.

In [ ]:
# --- Bootstrap (don't edit) ---
# Adds the project root to sys.path so we can do `from src.xxx import yyy`.
import sys, os
from pathlib import Path
ROOT = Path.cwd()
# Walk up until we find the project root (folder that contains src/)
for _ in range(4):
    if (ROOT / 'src').exists() and (ROOT / 'requirements.txt').exists():
        break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print('Project root:', ROOT)


## 5.1 — Build the store (one-time, ~15s)

In [ ]:
from src.rag_utils import build_store_from_folder, rag_answer
store = build_store_from_folder('data/generated/pdf', chunk_size=800, overlap=120)
print(f'Store has {len(store)} chunks.')

## 5.2 — Ask a CA-typical question

In [ ]:
question = 'What are the key audit risks identified in the planning memo?'
answer, hits = rag_answer(question, store, k=4, return_hits=True)
print('QUESTION:', question)
print('\n--- ANSWER ---')
print(answer)
print('\n--- RETRIEVED CHUNKS ---')
for h in hits:
    print(f"  {h['metadata'].get('source')} p.{h['metadata'].get('page')}  score={h['score']:.3f}")

## 5.3 — A batch of CA questions

In [ ]:
ca_questions = [
    'Which transactions look unusual based on the audit memos?',
    'What does the procurement policy say about approvals for purchases above NPR 5 lakh?',
    'Are there any possible compliance issues with related parties?',
    'Summarise the related party transactions disclosed in the annual report.',
    'What is the loan covenant for DSCR and is it being met?',
]
for q in ca_questions:
    print('Q:', q)
    print('A:', rag_answer(q, store, k=4))
    print('-' * 80)

## What just happened?

For each question, the system:
1. Embedded the question.
2. Found the top-4 most similar chunks from your PDFs.
3. Built a prompt: "Answer this question using only the context below: [chunks]".
4. Sent it to the LLM and returned the answer with citations.


## Expected output

Answers should quote the planning memo's five risks, the policy's three-tier approval matrix, and the loan agreement's 1.25x DSCR covenant. Each fact should be tagged with the source filename and page number.

## Exercise

1. Ask: *"What are the policy rules for related-party purchases?"* — does the answer combine the Procurement Policy and the NFRS policy note?
2. Reduce `k` to 1 and re-ask the audit-risk question. What is missing?
3. Increase `k` to 8. Does the answer get better or noisier?


## Common errors

| Symptom | Fix |
|---|---|
| Answer says "I could not find this" | The information isn't in the docs *or* retrieval missed. Try rephrasing the question or raising `k`. |
| Answer cites a wrong page | Inspect the retrieved chunks — the model may be paraphrasing across two chunks. |
| Answer invents facts | Bad sign — try a stricter system prompt ("Use ONLY the context. Quote verbatim where possible.") |


## ⚠️ Professional caution

RAG is **not** "AI that is always right". It is "AI that is *less likely* to hallucinate, because we forced it to look at our text first". Always check the cited chunk yourself before relying on the answer for client work.